# Cvičné úlohy — řešení

Ke každé úloze: seznam chyb, opravený kód, rozšíření a výsledky ladění.
**Dívej se sem až po vlastním pokusu** — jinak si vyrobíš pocit znalosti bez znalosti.

---

## Úloha 1 — Průměrná známka

*Archetyp: chybějící return, dělení nulou (katalog #3, #13)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Chybí `return`.** Funkce průměr spočítá, uloží do `vysledek` a zahodí — vrací `None`. |
| 2 | runtime | Následkem toho `f"{p:.2f}"` spadne: `TypeError: unsupported format string passed to NoneType`. |
| 3 | runtime | `soucet / len(znamky)` spadne na `ZeroDivisionError` u prázdného seznamu. |

**Co kód udělá:** spadne na `TypeError` při formátování. Kdyby tam bylo jen `print(p)`,
vypsalo by `None` a chyba by se dala přehlédnout.

### Opravené a rozšířené řešení

In [ ]:
def prumer(znamky, zaokrouhli=2):
    """
    Vrací průměr známek zaokrouhlený na daný počet míst.

    Vyhazuje ValueError pro prázdný seznam nebo známku mimo rozsah 1-5.
    """
    if not isinstance(znamky, list):
        raise TypeError(f"očekávám seznam, dostal jsem {type(znamky).__name__}")
    if not znamky:
        raise ValueError("prázdný seznam — není z čeho počítat průměr")

    for i, z in enumerate(znamky):
        if isinstance(z, bool) or not isinstance(z, int):
            raise ValueError(f"známka na indexu {i} není celé číslo: {z!r}")
        if not 1 <= z <= 5:
            raise ValueError(f"známka na indexu {i} je mimo rozsah 1-5: {z}")

    return round(sum(znamky) / len(znamky), zaokrouhli)


def slovni_hodnoceni(prumer_znamek):
    """Vrací slovní hodnocení podle průměru. Pozor na pořadí podmínek!"""
    if prumer_znamek <= 1.5:
        return "výborný"
    elif prumer_znamek <= 2.5:
        return "chvalitebný"
    elif prumer_znamek <= 3.5:
        return "dobrý"
    elif prumer_znamek <= 4.5:
        return "dostatečný"
    return "nedostatečný"


trida = [1, 2, 1, 3, 2]
p = prumer(trida)
print(f"Průměr třídy: {p:.2f} ({slovni_hodnoceni(p)})")   # 1.80 (chvalitebný)
print(prumer(trida, zaokrouhli=0))                        # 2

### Výsledky ladění

- **Funguje:** `[1,2,1,3,2]` → součet 9, děleno 5 = `1.8`. Ověřeno ručně.
- **Funguje:** `zaokrouhli=0` vrací `2` — `round` zaokrouhluje na nejbližší.
- **Funguje:** `slovni_hodnoceni(1.8)` → `'chvalitebný'`, protože 1.8 je nad 1.5 a pod 2.5.
- **Hraniční:** prázdný seznam → `ValueError` místo `ZeroDivisionError`.
  Jednoprvkový seznam → průměr je ta známka.
- **Hraniční:** známka `0`, `6` nebo `"1"` → `ValueError` s indexem.
- **Omezení:** `bool` je odmítnut zvlášť — `True` by jinak prošlo jako známka 1.
- **Omezení:** `round` používá bankovní zaokrouhlování (`round(2.5)` je `2`, ne `3`).
  U známek to nevadí, ale je dobré to vědět.

### Na co se doptají

- **Co vrací funkce bez `return`?** `None`. Proto `p + 1` nebo `f"{p:.2f}"` spadne na `TypeError`.
- **Proč `ValueError` a ne návratová hodnota `None` u prázdného seznamu?** Průměr prázdné množiny
  není definovaný — je to chybový stav, ne legitimní výsledek.
- **Proč jde pořadí `elif` od nejmenšího?** Protože podmínky jsou `<=`. Kdyby byly `>=`,
  muselo by se jít od největšího. Obrácené pořadí udělá nedosažitelné větve.
- **Jaký je rozdíl mezi `round(x, 2)` a `f"{x:.2f}"`?** `round` vrací **číslo**,
  f-string **řetězec**. Na výpočty round, na výpis f-string.

---

## Úloha 2 — Odhad splátek půjčky

*Archetyp: nekonečný while (katalog #6)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **`dluh - splatka` nic nedělá.** Výsledek se zahodí, musí být `dluh -= splatka`. |
| 2 | sémantická | **`mesice` se nikdy nezvýší** — chybí `mesice += 1`. |
| 3 | sémantická | Kombinace obou → `dluh` jen **roste** o úrok → `while dluh > 0` je **nekonečný**. |
| 4 | návrhová | Chybí ochrana proti splátce menší než úrok — i po opravě by se zacyklilo. |

**Nejzákeřnější na tom je, že to nespadne** — buňka prostě běží dál a dál. V Jupyteru
poznáš zacyklení podle toho, že vedle buňky zůstává `[*]`. Řešení: **Interrupt kernel**.

### Opravené a rozšířené řešení

In [ ]:
def pocet_splatek(dluh, splatka, urok_mesicne=0.01, max_mesicu=600):
    """
    Vrací (počet měsíců do splacení, celkem zaplaceno).

    Vyhazuje ValueError, je-li splátka tak nízká, že dluh nikdy nesplatíš.
    """
    if dluh <= 0:
        raise ValueError(f"dluh musí být kladný, dostal jsem {dluh}")
    if splatka <= 0:
        raise ValueError(f"splátka musí být kladná, dostal jsem {splatka}")

    # klíčová pojistka: první měsíční úrok musí být menší než splátka
    prvni_urok = dluh * urok_mesicne
    if splatka <= prvni_urok:
        raise ValueError(
            f"splátka {splatka} nepokryje ani měsíční úrok {prvni_urok:.2f} "
            f"— dluh by rostl donekonečna"
        )

    mesice = 0
    zaplaceno = 0.0
    while dluh > 0:
        dluh = dluh * (1 + urok_mesicne)   # nabíhá úrok
        splaceno_ted = min(splatka, dluh)  # poslední splátka bývá nižší
        dluh -= splaceno_ted               # oprava 1: přiřazení
        zaplaceno += splaceno_ted
        mesice += 1                        # oprava 2: posun počítadla

        if mesice > max_mesicu:            # pojistka proti zacyklení
            raise ValueError(f"nesplaceno ani za {max_mesicu} měsíců")

    return mesice, round(zaplaceno, 2)


mesice, celkem = pocet_splatek(10000, 500)
print(f"Splaceno za {mesice} měsíců, celkem zaplaceno {celkem} Kč")
print(f"Přeplatek: {celkem - 10000:.2f} Kč")

In [ ]:
# Chybové stavy
for popis, args in [
    ("splátka nižší než úrok", (10000, 50)),
    ("nulový dluh",            (0, 500)),
    ("záporná splátka",        (10000, -5)),
]:
    try:
        pocet_splatek(*args)
        print(f"{popis:24}: PROŠLO (nemělo!)")
    except ValueError as e:
        print(f"{popis:24}: {e}")

### Výsledky ladění

- **Funguje:** 10 000 Kč při splátce 500 a úroku 1 % měsíčně → 23 měsíců,
  celkem 11 213.48 Kč. Přeplatek 1 213.48 Kč, což u 1 % měsíčně sedí řádově.
- **Funguje:** poslední splátka je nižší než 500 — proto `min(splatka, dluh)`,
  jinak by se přeplatilo a `zaplaceno` by nesedělo.
- **Funguje:** splátka 50 při úroku 100 Kč/měsíc → `ValueError` **hned**, bez zacyklení.
- **Hraniční:** nulový nebo záporný dluh i splátka → `ValueError`.
- **Pojistka:** `max_mesicu=600` (50 let) — kdyby kontrola úroku něco propustila,
  cyklus stejně skončí. **Dvojitá ochrana je u `while` dobrý zvyk.**
- **Omezení:** úrok se počítá před splátkou, což odpovídá běžné praxi u hypoték.
  Opačné pořadí by dalo o něco nižší přeplatek.
- **Omezení:** `float` aritmetika kumuluje nepřesnost. Pro reálné peníze by se použil `Decimal`.

### Na co se doptají

- **Jak poznáš nekonečný cyklus v Jupyteru?** Buňka zůstane s `[*]` a nikdy nedoběhne.
  Zastavíš ji přes Interrupt kernel (⏹ nebo dvakrát `I`).
- **Proč `dluh - splatka` nic neudělá?** Je to jen výraz, jeho hodnota se zahodí.
  Přiřazení je `dluh = dluh - splatka`, zkráceně `dluh -= splatka`.
- **Kdy `while` a kdy `for`?** `while` když nevíš počet iterací dopředu — přesně tenhle případ.
  Kdybys věděl, že to je 22 měsíců, dal bys `for`.
- **Jak se obecně bránit zacyklení?** Vždy si ověř, že se **řídicí proměnná uvnitř mění směrem
  k ukončení**. U výpočtů přidej tvrdý limit iterací.

---

## Úloha 3 — Zařazení do věkové kategorie

*Archetyp: pořadí elif, nedosažitelná větev (katalog #7)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Pořadí `elif` je od nejmenší hranice.** První podmínka `vek >= 6` chytí všechny od 6 výš, takže větve pro 15, 19 a 60 jsou **nedosažitelné**. |
| 2 | návrhová | Chybí ošetření záporného věku — `-5` vrátí „předškolák". |

**Co kód udělá:** 16, 25 i 65 let vrátí `"žák"`. Kód běží, nespadne a **je úplně špatně**.

**Pravidlo:** u `if/elif` s nerovnostmi jdi vždy **od nejužší podmínky k nejširší** —
u `>=` tedy od největší hranice, u `<=` od nejmenší.

### Opravené a rozšířené řešení

In [ ]:
VYCHOZI_HRANICE = {
    "veterán": 60,
    "dospělý": 19,
    "dorostenec": 15,
    "žák": 6,
}


def kategorie(vek, hranice=None):
    """
    Vrací sportovní kategorii podle věku.

    hranice — slovník kategorie -> minimální věk (výchozí VYCHOZI_HRANICE)

    Vyhazuje ValueError pro záporný věk, TypeError pro nečíselný vstup.
    """
    if isinstance(vek, bool) or not isinstance(vek, (int, float)):
        raise TypeError(f"věk musí být číslo, dostal jsem {type(vek).__name__}")
    if vek < 0:
        raise ValueError(f"věk nemůže být záporný: {vek}")

    if hranice is None:                 # NIKDY měnitelný default!
        hranice = VYCHOZI_HRANICE

    # seřadit sestupně podle hranice -> první, která platí, je ta správná
    for nazev, minimum in sorted(hranice.items(), key=lambda p: p[1], reverse=True):
        if vek >= minimum:
            return nazev
    return "předškolák"


for v in [4, 8, 16, 25, 65]:
    print(f"{v:3} let -> {kategorie(v)}")

In [ ]:
# Varianta s klasickým elif — SPRÁVNÉ pořadí, od největší hranice
def kategorie_elif(vek):
    if vek < 0:
        raise ValueError(f"věk nemůže být záporný: {vek}")
    if vek >= 60:
        return "veterán"
    elif vek >= 19:
        return "dospělý"
    elif vek >= 15:
        return "dorostenec"
    elif vek >= 6:
        return "žák"
    else:
        return "předškolák"


def kategorie_rozsah(nazev, hranice=None):
    """Vrací (od, do) pro danou kategorii; do je None u nejvyšší."""
    if hranice is None:
        hranice = VYCHOZI_HRANICE
    if nazev not in hranice:
        raise ValueError(f"neznámá kategorie: {nazev!r}")

    serazene = sorted(hranice.items(), key=lambda p: p[1])
    for i, (jmeno, od) in enumerate(serazene):
        if jmeno == nazev:
            do = serazene[i + 1][1] - 1 if i + 1 < len(serazene) else None
            return od, do


# obě varianty musí souhlasit
for v in range(0, 80):
    assert kategorie(v) == kategorie_elif(v), f"neshoda pro {v}"
print("obě varianty souhlasí pro věk 0-79")

for k in ["žák", "dorostenec", "dospělý", "veterán"]:
    print(f"{k:12} {kategorie_rozsah(k)}")

### Výsledky ladění

- **Funguje:** 4 → předškolák, 8 → žák, 16 → dorostenec, 25 → dospělý, 65 → veterán.
  Původní kód vracel „žák" pro všechny od 6 výš.
- **Funguje:** obě varianty (slovník i `elif`) souhlasí pro celý rozsah 0–79, ověřeno `assert`.
- **Funguje:** vlastní `hranice` umožní změnit pravidla bez zásahu do kódu funkce.
- **Hraniční:** přesně na hranici (6, 15, 19, 60) padne do **vyšší** kategorie —
  odpovídá `>=`. Věk 0 → předškolák.
- **Hraniční:** záporný věk → `ValueError`, `"25"` nebo `True` → `TypeError`.
- **Omezení:** `hranice=None` a přiřazení uvnitř je nutné — měnitelný default (slovník)
  by se sdílel mezi voláními.
- **Poznámka:** verze se slovníkem je odolnější (pořadí si dopočítá sama), verze s `elif`
  je čitelnější. U zkoušky bych ukázal obě a nechal komisi vybrat.

### Na co se doptají

- **Proč byly ty větve nedosažitelné?** `elif` se vyhodnotí jen když všechny předchozí
  podmínky neplatily. `vek >= 6` je nejširší, takže pohltí všechny ostatní.
- **Jak to obecně poznat?** U řetězce `elif` s nerovnostmi musí hranice tvořit **monotónní
  posloupnost** ve směru od nejužší k nejširší. Když ne, něco je nedosažitelné.
- **Proč `sorted(..., reverse=True)`?** Aby se testovalo od nejvyšší hranice — tím se
  problém s pořadím vyřeší **automaticky**, i když někdo do slovníku přidá kategorii.
- **Co dělá `key=lambda p: p[1]`?** Řadí dvojice podle druhé složky, tedy podle věkové hranice.
- **Proč `hranice=None` a ne `hranice=VYCHOZI_HRANICE`?** Měnitelný default se vytvoří jednou
  při definici; kdyby ho někdo uvnitř změnil, projeví se to ve všech dalších voláních.

---

## Úloha 4 — Výpis násobilky

*Archetyp: off-by-one v range (katalog #8)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **`range(1, n)` končí u `n-1`.** Pro `n=10` vypíše jen do 9× — chybí poslední řádek. Má být `range(1, n + 1)`. |

**Co kód udělá:** vypíše devět řádků místo deseti a vrátí `135` místo `165`.
Rozdíl je přesně `3 × 10 = 30`.

**Tohle je nejčastější chyba vůbec** a je zákeřná tím, že výsledek *vypadá* rozumně —
nepoznáš ji, dokud si ho nespočítáš nezávisle. Součet násobků 3 od 1 do 10 je
$3 \cdot (1 + 2 + \dots + 10) = 3 \cdot 55 = 165$.

### Opravené a rozšířené řešení

In [ ]:
def nasobilka(cislo, n=10, od=1, krok=1, vypisovat=True):
    """
    Vypíše násobilku čísla od `od` do `n` (včetně) s daným krokem.

    Vrací (součet vypsaných násobků, počet řádků).
    Vyhazuje ValueError pro nekladný krok nebo od > n.
    """
    if isinstance(cislo, bool) or not isinstance(cislo, (int, float)):
        raise TypeError(f"číslo musí být číslo, dostal jsem {type(cislo).__name__}")
    if krok < 1:
        raise ValueError(f"krok musí být kladný, dostal jsem {krok}")
    if od > n:
        raise ValueError(f"od ({od}) nesmí být větší než n ({n})")

    soucet = 0
    pocet = 0
    for i in range(od, n + 1, krok):        # oprava: n + 1, aby se zahrnulo n
        nasobek = cislo * i
        if vypisovat:
            print(f"{cislo} x {i} = {nasobek}")
        soucet += nasobek
        pocet += 1
    return soucet, pocet


soucet, pocet = nasobilka(3)
print(f"součet: {soucet}, řádků: {pocet}")     # 165, 10

In [ ]:
# Nezávislé ověření vzorcem: 3 * (1+2+...+10) = 3 * 55 = 165
ocekavany = 3 * sum(range(1, 11))
print(f"kontrola vzorcem: {ocekavany}")
assert nasobilka(3, vypisovat=False)[0] == ocekavany

# varianty
print(nasobilka(5, n=3, vypisovat=False))            # (5+10+15, 3) = (30, 3)
print(nasobilka(2, n=10, krok=2, vypisovat=False))   # 2*(2+4+6+8+10)=60, 5 řádků
print(nasobilka(4, od=5, n=7, vypisovat=False))      # 4*(5+6+7)=72, 3 řádky

for popis, kw in [("krok 0", {"krok": 0}), ("od > n", {"od": 20, "n": 5})]:
    try:
        nasobilka(3, vypisovat=False, **kw)
        print(f"{popis}: PROŠLO (nemělo!)")
    except ValueError as e:
        print(f"{popis}: {e}")

### Výsledky ladění

- **Funguje:** `nasobilka(3)` vypíše 10 řádků a vrátí součet `165`.
  Ověřeno **nezávisle vzorcem** $3 \cdot (1+2+\dots+10) = 3 \cdot 55 = 165$, ne opsáním z výstupu.
- **Funguje:** `n=3` → `(30, 3)`, tedy 5+10+15. `krok=2` → `(50, 5)`, tedy 2·(1+3+5+7+9) — krok se počítá
  **od `od=1`**, takže se berou liché násobky, ne sudé. `od=5, n=7` → `(72, 3)`.
- **Hraniční:** `od == n` vypíše jeden řádek. `n=1` vypíše jen `cislo x 1`.
- **Hraniční:** `krok=0` i `od > n` → `ValueError` (bez toho by `range` vrátil prázdno a tiše `0`).
- **Poznámka:** parametr `vypisovat` jsem přidal navíc, aby šly psát testy bez zaplavení výstupu.
  U zkoušky je to dobrý argument — funkce, která **jen tiskne**, se špatně testuje.
- **Omezení:** desetinné `cislo` funguje, ale výpis pak není zarovnaný.

### Na co se doptají

- **Proč `range(1, n)` nezahrne `n`?** `stop` je vždy **výlučný**. `range(1, 10)` je 1–9,
  tedy 9 hodnot. Konvence je zvolená tak, aby `range(len(s))` dalo přesně platné indexy.
- **Jak si to ověřit, aniž bys počítal řádky?** Spočítej výsledek **nezávisle** —
  vzorcem, v hlavě, na papíře. Nikdy neopisuj očekávanou hodnotu z výstupu programu.
- **Kolik prvků má `range(a, b, k)`?** `ceil((b - a) / k)` pro kladný krok.
- **Proč se testuje `krok < 1` a ne `krok == 0`?** Záporný krok by u `range(1, n+1, -1)`
  dal prázdnou posloupnost a funkce by tiše vrátila `(0, 0)` — tiché selhání je horší než výjimka.

---

## Úloha 5 — Zápis do docházky

*Archetyp: mutable default argument (katalog #10)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **Mutable default argument.** Seznam `[]` se vytvoří **jednou při definici funkce**, ne při každém volání. Všechna volání bez seznamu pak sdílejí tentýž objekt. |

**Co kód udělá:** úterý vypíše `['Anna', 'Bob']` — Anna se propašuje do úterní docházky.
Navíc `pondeli is utery` je `True`, takže se změnil i pondělní seznam.

**Proč je to zrádné:** vypadá to naprosto nevinně a čtením se to skoro nedá odhalit.
Ověříš to takhle: `print(zapis.__defaults__)` — uvidíš seznam, který mezi voláními roste.

### Opravené a rozšířené řešení

In [ ]:
def zapis(jmeno, dochazka=None):
    """
    Zapíše jméno do docházky (bez duplicit). Když seznam není zadán, založí NOVÝ.
    Vrací docházku.
    """
    if not isinstance(jmeno, str) or not jmeno.strip():
        raise ValueError(f"jméno musí být neprázdný řetězec, dostal jsem {jmeno!r}")

    if dochazka is None:          # ← klíčová oprava
        dochazka = []

    jmeno = jmeno.strip()
    if jmeno not in dochazka:     # bez duplicit
        dochazka.append(jmeno)
    return dochazka


def odepis(dochazka, jmeno):
    """Odstraní jméno z docházky. Neznámé jméno -> ValueError."""
    if jmeno not in dochazka:
        raise ValueError(f"{jmeno!r} v docházce není")
    dochazka.remove(jmeno)
    return dochazka


def souhrn(dochazka):
    """Formátovaný přehled docházky."""
    if not dochazka:
        return "Nikdo není přítomen."
    return f"Přítomno {len(dochazka)}: {', '.join(sorted(dochazka))}"

In [ ]:
pondeli = zapis("Anna")
utery = zapis("Bob")
print("pondělí:", pondeli)                    # ['Anna']
print("úterý:  ", utery)                      # ['Bob']  ← už se nemíchají
print("sdílejí objekt?", pondeli is utery)    # False

zapis("Cyril", pondeli)
zapis("Anna", pondeli)          # duplicita — nepřidá se
zapis("  Dana  ", pondeli)      # mezery se oříznou
print(souhrn(pondeli))          # Přítomno 3: Anna, Cyril, Dana

odepis(pondeli, "Cyril")
print(souhrn(pondeli))          # Přítomno 2: Anna, Dana
print(souhrn([]))               # Nikdo není přítomen.

for popis, volani in [
    ("prázdné jméno",  lambda: zapis("   ")),
    ("není řetězec",   lambda: zapis(42)),
    ("neznámé jméno",  lambda: odepis(pondeli, "Emil")),
]:
    try:
        volani()
        print(f"{popis:16}: PROŠLO (nemělo!)")
    except ValueError as e:
        print(f"{popis:16}: {e}")

### Výsledky ladění

- **Funguje:** dvě samostatná volání bez docházky vrací **nezávislé** seznamy —
  ověřeno `is`, vrací `False`. Tím je původní chyba prokazatelně opravená.
- **Funguje:** duplicita se nepřidá — `zapis("Anna", pondeli)` podruhé seznam nezmění.
- **Funguje:** `"  Dana  "` se uloží jako `"Dana"` (ořez mezer), takže `" Dana"` a `"Dana"`
  se považují za totéž.
- **Funguje:** `souhrn` řadí abecedně a prázdnou docházku hlásí zvlášť.
- **Hraniční:** prázdné jméno i samé mezery → `ValueError`. Odepsání neznámého → `ValueError`.
- **Omezení:** duplicity se hledají **case-sensitive** — `"anna"` a `"Anna"` jsou dvě různá jména.
  Šlo by dořešit porovnáním přes `.lower()`, zadání to nežádalo.
- **Omezení:** `souhrn` řadí podle Unicode, takže česká diakritika skončí za nediakritickými znaky.

### Na co se doptají

- **Proč se výchozí hodnota vytvoří jen jednou?** `def` je příkaz, který se vykoná jednou
  při definici. Výchozí hodnoty se tehdy vyhodnotí a uloží do `funkce.__defaults__`.
- **Které typy jsou v defaultu nebezpečné?** Všechny **měnitelné**: `list`, `dict`, `set`.
  Neměnitelné (`int`, `str`, `tuple`, `None`) jsou v pořádku.
- **Jak to ověřit?** `print(zapis.__defaults__)` — u chybné verze uvidíš rostoucí seznam.
- **Proč `is None` a ne `== None`?** `is` porovnává identitu a nedá se přebít vlastním `__eq__`.
- **Dá se toho využít schválně?** Ano, jako primitivní cache mezi voláními — ale je to matoucí,
  lepší je `functools.lru_cache`.

---

## Úloha 6 — Hledání odmocniny půlením intervalu

*Archetyp: porovnání float, while bez tolerance (katalog #11, #6)*

### Nalezené chyby

| # | Typ | Popis |
|---|-----|-------|
| 1 | sémantická | **`stred * stred != cislo` porovnává `float` na přesnou rovnost.** Pro `√2` se nikdy netrefí → **nekonečný cyklus**. Musí se porovnávat s **tolerancí**. |
| 2 | sémantická | **Pro `cislo < 1` je špatný interval.** `√0.25 = 0.5`, ale horní mez je `0.25` — hledaná hodnota v intervalu vůbec neleží. |
| 3 | návrhová | Záporný vstup dá nesmyslný interval a zacyklí se. |

**Co kód udělá:** `odmocnina(16)` náhodou projde (4 je přesně reprezentovatelná),
`odmocnina(2)` se **zacyklí navždy**.

Tohle je nejrealističtější úloha z celé šestice — přesně takhle vypadá numerický kód
napsaný bez rozmyslu. Souvisí s [SZZTP okruh 4](../../../SZZTP/04-funkce-polynomy-nelinearni-rovnice/).

### Opravené a rozšířené řešení

In [ ]:
import math


def odmocnina(cislo, tolerance=1e-9, max_iteraci=200):
    """
    Najde odmocninu metodou půlení intervalu.

    Vrací (odmocnina, počet iterací).
    Vyhazuje ValueError pro záporné číslo nebo při nedosažení tolerance.
    """
    if isinstance(cislo, bool) or not isinstance(cislo, (int, float)):
        raise TypeError(f"očekávám číslo, dostal jsem {type(cislo).__name__}")
    if cislo < 0:
        raise ValueError(f"nelze odmocnit záporné číslo: {cislo}")
    if cislo == 0:
        return 0.0, 0

    dolni = 0.0
    horni = max(cislo, 1.0)      # oprava 2: pro cislo < 1 je odmocnina VĚTŠÍ než cislo

    for iterace in range(1, max_iteraci + 1):
        stred = (dolni + horni) / 2
        rozdil = stred * stred - cislo

        if abs(rozdil) < tolerance:      # oprava 1: tolerance místo přesné rovnosti
            return stred, iterace
        if rozdil < 0:
            dolni = stred
        else:
            horni = stred

    raise ValueError(f"nedosaženo tolerance {tolerance} ani za {max_iteraci} iterací")

In [ ]:
# porovnání s math.sqrt
print(f"{'číslo':>8} {'moje':>18} {'math.sqrt':>18} {'iterací':>8}  shoda")
for x in [16, 2, 0.25, 1, 0, 100, 1e6]:
    moje, iteraci = odmocnina(x)
    presne = math.sqrt(x)
    shoda = math.isclose(moje, presne, abs_tol=1e-6)
    print(f"{x:>8} {moje:>18.10f} {presne:>18.10f} {iteraci:>8}  {shoda}")

print()
for popis, arg in [("záporné", -4), ("není číslo", "16")]:
    try:
        odmocnina(arg)
        print(f"{popis:12}: PROŠLO (nemělo!)")
    except (ValueError, TypeError) as e:
        print(f"{popis:12}: {type(e).__name__}: {e}")

### Výsledky ladění

- **Funguje:** `√16 = 4.0`, `√2 ≈ 1.4142135624`, `√100 = 10.0` — všechny se shodují
  s `math.sqrt` na 6 desetinných míst (ověřeno `math.isclose`).
- **Funguje:** `√0.25 = 0.5` — to je vstup, na kterém původní kód selhává,
  protože hledaná hodnota leží **nad** horní mezí. Oprava `max(cislo, 1.0)` to řeší.
- **Funguje:** `√2` doběhne za 30 iterací místo zacyklení. Půlení intervalu má
  **lineární konvergenci** — každá iterace přidá zhruba jeden bit přesnosti,
  takže na $10^{-9}$ je potřeba asi $\log_2(10^9) \approx 30$ iterací. Sedí přesně.
- **Pozor na `√1`:** vrací `0.9999999995`, ne přesně `1.0` — tolerance se testuje na
  **druhé mocnině**, takže výsledek je přesný jen na $\sqrt{\text{tolerance}}$.
- **Hraniční:** `0` vrací `(0.0, 0)` bez iterování. `1` vrací `1.0`.
- **Hraniční:** záporné číslo → `ValueError`, řetězec → `TypeError`.
- **Pojistka:** `max_iteraci=200` je bohatě nad potřebou — kdyby tolerance byla
  nesmyslně malá, vyhodí `ValueError` místo zacyklení.
- **Omezení:** pro velmi velká čísla (`1e300`) by tolerance `1e-9` byla nedosažitelná
  v absolutním vyjádření — správně by se použila **relativní** tolerance.

### Na co se doptají

- **Proč se nesmí `float` porovnávat na rovnost?** Binární plovoucí čárka neumí přesně
  vyjádřit většinu desetinných čísel. `√2` je navíc iracionální — přesnou rovnost nikdy netrefíš.
- **Jak se to dělá správně?** `abs(a - b) < tolerance`, nebo `math.isclose(a, b)`.
- **Proč `max(cislo, 1.0)` jako horní mez?** Pro `x < 1` platí `√x > x`
  (např. `√0.25 = 0.5`), takže interval `[0, x]` hledanou hodnotu neobsahuje.
- **Kolik iterací to potřebuje?** Interval se každou iterací **půlí**, takže na přesnost
  $\varepsilon$ je potřeba $\log_2(\text{rozsah}/\varepsilon)$ iterací — logaritmicky málo.
- **Znáš rychlejší metodu?** Newtonova — má **kvadratickou** konvergenci (počet správných
  číslic se každou iterací zdvojnásobí), ale potřebuje derivaci a nemusí konvergovat vždy.
  Podrobně v [SZZTP okruh 4](../../../SZZTP/04-funkce-polynomy-nelinearni-rovnice/).
- **Proč `for` s limitem místo `while`?** Dostaneš pojistku proti zacyklení zadarmo —
  u numerických metod je to dobrý zvyk.

---